In [30]:
"""
CS4771 - Python for Machine Learning
Kaggle Competition Assignment 2
Todd Carter
V01187982
10-22-2025

"""
#
# The goal of this competition is to predict whether a 
# flight will be delayed by more than 15 minutes 
# using multiple types of features.
#
# XGBoost was used for modeling.
#

# Import essential libraries
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score, f1_score

# Comparison models
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression

train = pd.read_csv("/kaggle/input/competition-datasets-reupload/train.csv")
test = pd.read_csv("/kaggle/input/competition-datasets-reupload/test.csv")
sample = pd.read_csv("/kaggle/input/competition-datasets-reupload/submission.csv")

# Print headers to see the data types:
print("\ntrain header is: \n", train.head())
print("\ntrain info is: \n", train.info())
print("\ntest header is: \n", test.head())
print("\ntest info is: \n", test.info())
print("\sample header is: \n", sample.head())

# Extract time of day from Scheduled_DEP and Actual_ARR_dt_Ori:
for col in ["Scheduled_DEP", "Actual_ARR_dt_Ori"]:
    train[col] = pd.to_datetime(train[col], errors="coerce")
    test[col] = pd.to_datetime(test[col], errors="coerce")
    train[col + "_HOUR"] = train[col].dt.hour
    test[col + "_HOUR"] = test[col].dt.hour

# Extract actual arrival day:
for col in ["Actual_ARR_dt_Ori"]:
    train["ACTUAL_ARRIVAL_DAY"] = train[col].dt.day
    test["ACTUAL_ARRIVAL_DAY"] = test[col].dt.day

# Dropped columns are mostly repeated date information
X = train.drop(columns = [
    'FL_DATE', 'CRS_DEP_1hrpre', 'CRS_DEP_1hrpost', 
    'Scheduled_DEP', 'Scheduled_ARR_Ori', 'Actual_ARR_dt_Ori',
    'Scheduled_DEP_EST', 'Scheduled_ARR_EST', 'Scheduled_ARR_Local',
    "DEP_DEL15",
    'DEP_1hrpre_num', 'Arr_1hrpre_num'
])

y = train["DEP_DEL15"]

X_test = test.drop(columns=[
    'FL_DATE', 'CRS_DEP_1hrpre', 'CRS_DEP_1hrpost', 
    'Scheduled_DEP', 'Scheduled_ARR_Ori', 'Actual_ARR_dt_Ori',
    'Scheduled_DEP_EST', 'Scheduled_ARR_EST', 'Scheduled_ARR_Local',
    'DEP_1hrpre_num', 'Arr_1hrpre_num'
])

# Print headers to see if encoding worked:
print("\nX header is: \n", X.head())
print("\X info is: \n", X.info())

# Define column types:
numerical_features = X.select_dtypes(include=np.number).columns.tolist()
categorical_features = X.select_dtypes(include=['object']).columns.tolist()

print(f"Numerical Features: {numerical_features}")
print(f"Categorical Features: {categorical_features}")

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_features), # Scale numerical features
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features) # One-hot encode categorical features
    ],
    remainder='passthrough'
)

# Split data:
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f"Training set size: {X_train.shape[0]} samples")

# Modeling and fitting:
# Calculate scale_pos_weight for imbalance handling
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

xgb_baseline = xgb.XGBClassifier(
    objective='binary:logistic',
    eval_metric='auc',
    use_label_encoder=False,
    scale_pos_weight=scale_pos_weight,
    random_state=42
)

pipeline_baseline = Pipeline(steps=[('preprocessor', preprocessor), ('classifier', xgb_baseline)])

pipeline_baseline.fit(X_train, y_train)

y_pred_baseline = pipeline_baseline.predict(X_val)

y_prob_baseline = pipeline_baseline.predict_proba(X_test)[:, 1]

# Convert into binary to be read by F1-scoring:
y_prob_baseline_binary = (y_prob_baseline >= 0.5).astype(int)

# Produce the submission csv:
submission = pd.DataFrame({
    "ID": test["ID"],
    "DEP_DEL15": y_prob_baseline_binary
})

submission.to_csv("submission.csv", index=False)


train header is: 
    MONTH  DAY_OF_MONTH     FL_DATE MKT_CARRIER OP_CARRIER ORIGIN DEST  \
0     10            12  2023-10-12          DL         DL    ATL  LAS   
1      7            20  2023-07-20          DL         DL    ATL  ROA   
2      8            26  2023-08-26          AA         AA    SAN  PHL   
3     10             9  2023-10-09          AA         AA    ORD  PHL   
4     11            30  2023-11-30          UA         OO    SFO  TUS   

   CRS_ELAPSED_TIME  DISTANCE    CRS_DEP_1hrpre  ... scheduled_Turnarnd  \
0             267.0      1747  12OCT23:20:00:00  ...               70.0   
1              83.0       357  20JUL23:22:00:00  ...               65.0   
2             318.0      2370  26AUG23:08:00:00  ...               51.0   
3             122.0       678  09OCT23:07:00:00  ...               61.0   
4             137.0       751  30NOV23:08:00:00  ...               80.0   

         Scheduled_DEP    Scheduled_ARR_Ori    Actual_ARR_dt_Ori  \
0  10/12/2023 21:55:00